# 독성 예측 — 재현 가능한 평가 읽기

기존 LightGBM 모델 방향을 유지하고 데이터 품질과 평가 절차를 개선했습니다.
원본 데이터 없이 저장소에 포함된 **집계 결과만** 확인할 수 있습니다. 이 노트북은 학습을 자동 실행하지 않습니다.

**0=독성, 1=비독성.** 공식 F1과 독성 탐지 F1은 다릅니다.
이전에 탐색·제출한 교육 데이터의 후향적 검증이며 새로운 독립 테스트가 아닙니다.


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
REPORT = ROOT / 'reports' / 'retrospective-v1.json'
if not REPORT.exists():
    raise FileNotFoundError('저장소 루트에서 노트북을 실행하세요.')
report = json.loads(REPORT.read_text())
assert report['status'] == 'complete'
display(pd.Series(report['label_definition'], name='label meaning').to_frame())


## 1. 데이터와 평가 원칙

- 8,349행의 수치 입력은 ECFP/FCFP/PTFP 각각 1,024개와 Descriptor 4개입니다.
- 같은 chirality-free Murcko 골격 **또는** 동일 수치 입력을 전이적으로 묶어 fold 간 중복을 막습니다.
- 바깥 5-fold는 평가에만 사용합니다. 안쪽 3-fold에서 임계값과 LightGBM 반복 횟수를 결정합니다.
- 상수 제거와 선형 모델 스케일링은 각 학습 fold에서만 fit합니다.
- 전체 label을 참고해 만든 Morgan8 특성은 이번 실행에서 제외합니다.
- 골격 분리로 모든 분자 유사성·염·호변이성질체까지 통제한 것은 아닙니다.


In [ ]:
quality = report['data_quality']
display(pd.Series({k: quality[k] for k in ['rows', 'features', 'missing_cells', 'constant_features_descriptive_only', 'duplicate_smiles_extra', 'scaffold_groups', 'combined_groups']}).to_frame('count'))
display(pd.DataFrame({k: quality[k] for k in ['canonical_stereo_identity', 'connectivity_identity', 'exact_model_inputs']}))
display(pd.Series(quality['prediction_overlap']).to_frame('prediction overlap'))


## 2. 바깥 평가 결과

LightGBM은 기존 실험을 근거로 미리 정한 주 모델이며 Logistic Regression은 비교 기준입니다.
이번 바깥 결과를 보고 모델 종류·특성·설정을 다시 선택하지 않았습니다.
임계값은 비독성 F1 목표로 정했으며 실제 독성 안전성을 충족하는 기준이 아닙니다.


In [ ]:
rows = []
for model in report['models']:
    rows.append({'model': model['model'], **{key: model['pooled_outer_oof'][key] for key in ['f1_non_toxic_label1', 'f1_toxic_label0', 'recall_toxic_label0', 'roc_auc_non_toxic_label1', 'toxic_missed_as_non_toxic']}})
display(pd.DataFrame(rows).set_index('model'))
primary = report['models'][0]
display(pd.DataFrame(primary['pooled_outer_oof']['confusion_matrix_rows_actual_columns_predicted_0_1'], index=['실제 독성(0)', '실제 비독성(1)'], columns=['예측 독성(0)', '예측 비독성(1)']))


## 3. 목표에 따라 달라지는 임계값

고정 0.5와 안쪽 F1 최적화 임계값의 차이를 함께 확인합니다.
0.5 비교는 사전에 정한 진단용이며, 결과를 보고 최종 모델 정책을 자동 변경하지 않습니다.


In [ ]:
keys = ['f1_non_toxic_label1', 'f1_toxic_label0', 'recall_toxic_label0', 'toxic_missed_as_non_toxic']
display(pd.DataFrame({
    '안쪽 비독성 F1 선택': {k: primary['pooled_outer_oof'][k] for k in keys},
    '고정 0.5 비교': {k: primary['fixed_threshold_0_5_comparison'][k] for k in keys},
}))
display(pd.DataFrame(primary['error_slices']).T)


## 4. 과거 리더보드와 이번 실행 구분

과거 최고 F1은 4차 제출의 0.8438입니다. 제출 모델과 파일의 대응 관계는 확인되지 않았습니다.
학습 데이터와 예측 입력이 구조·수치 입력을 공유하므로 새로운 분자에 대한 독립 성능으로 소개하지 않습니다.
임계값·모델 설정·분리·학습량이 달라 이번 결과와의 차이를 하나의 원인으로 단정할 수 없습니다.


In [ ]:
history = json.loads((ROOT/'reports/historical-leaderboard.json').read_text())
display(pd.DataFrame(history['submissions']))
display(pd.Series(report['final_artifact']).to_frame('final local artifact'))


## 재실행

권한이 있는 원본 CSV를 별도로 준비하고 README의 `python -m toxicity.train train` 명령을 실행합니다.
전체 데이터 재학습 모델의 점수는 위의 바깥 OOF 성능과 동일한 측정 대상이 아닙니다.
모델·개별 예측·원본·파일 해시는 로컬 `artifacts/`에만 저장하며 Git에 포함하지 않습니다.
